# LogSAD — Setup & Evaluation Results
**Group 6 | Introduction to Deep Learning**

Paper: *Towards Training-free Anomaly Detection with Vision and Language Foundation Models* (CVPR 2025)

---
### Training Status: **Category D — Training-free**
LogSAD uses **pre-trained frozen foundation models** — no gradient updates, no fine-tuning.
- CLIP ViT-L/14 (DataComp-1B)
- DINOv2 ViT-L/14
- SAM ViT-H

The only "setup" steps are:
1. GPT-4V generates Match-of-Thought proposals **offline** (one-time, already done)
2. Memory bank is built from normal reference images at inference time

In [ ]:
import os, sys
# Run from LogSAD root so all local modules are importable
ROOT = os.path.abspath('..')
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('Working directory:', os.getcwd())

## 1. Foundation Model Loading
Three frozen models are loaded at startup. No weights are trained.

In [ ]:
import torch
import open_clip_local as open_clip
from dinov2.dinov2.hub.backbones import dinov2_vitl14
from segment_anything import sam_model_registry

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# CLIP ViT-L/14  (DataComp-1B pre-trained)
model_clip, _, _ = open_clip.create_model_and_transforms(
    'hf-hub:laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K'
)
model_clip.eval()
clip_params = sum(p.numel() for p in model_clip.parameters()) / 1e6
print(f'CLIP ViT-L/14      : {clip_params:.0f}M params  (frozen)')

# DINOv2 ViT-L/14
model_dinov2 = dinov2_vitl14()
model_dinov2.eval()
dino_params = sum(p.numel() for p in model_dinov2.parameters()) / 1e6
print(f'DINOv2 ViT-L/14    : {dino_params:.0f}M params  (frozen)')

# SAM ViT-H
sam = sam_model_registry['vit_h'](checkpoint='./checkpoint/sam_vit_h_4b8939.pth')
sam_params = sum(p.numel() for p in sam.parameters()) / 1e6
print(f'SAM ViT-H          : {sam_params:.0f}M params  (frozen)')

print(f'\nTotal              : {clip_params+dino_params+sam_params:.0f}M params')
print('All models frozen — NO training loop, NO backpropagation.')

## 2. Match-of-Thought (MoT) — juice_bottle

GPT-4V was used **offline (one-time)** to generate:
- **Interests of thought** — object parts to segment with SAM+CLIP
- **Compositional rules of thought** — logical constraints to check

This replaces manual annotation of logical rules.

In [ ]:
# Match-of-Thought proposals for juice_bottle (from paper Table 5)
mot_juice_bottle = {
    'interests_of_thought': [
        'glass',
        'liquid in bottle',
        'fruit',
        'label',
        'black background',
    ],
    'compositional_rules_of_thought': [
        'consistency of fruit tag and liquid color'
    ],
    'anomaly_free_caption': [
        'cherry juice bottle (red liquid + cherry tag)',
        'orange juice bottle (yellow liquid + orange tag)',
        'banana juice bottle (milky liquid + banana tag)',
    ],
    'liquid_color_prompts': ['red liquid', 'yellow liquid', 'milky liquid'],
    'fruit_tag_prompts':    ['cherry', 'orange', 'banana'],
}

print('=== juice_bottle Match-of-Thought ===')
for key, val in mot_juice_bottle.items():
    print(f'\n[{key}]')
    if isinstance(val, list):
        for v in val:
            print(f'  • {v}')
    else:
        print(f'  {val}')

print('\n→ Logical anomaly = liquid color does NOT match the fruit tag')

## 3. Memory Bank — Full-data Protocol

For the **full-data** protocol, patch features of all normal training images
are pre-computed with `compute_coreset.py` and stored as `.pt` files.

At inference time, each test patch is matched against this memory bank
(nearest-neighbour search) to compute an anomaly score.

In [ ]:
import torch

# Load pre-computed memory bank for juice_bottle (full-data)
clip_bank  = torch.load('memory_bank/mem_patch_feature_clip_juice_bottle.pt',
                        map_location='cpu')
dino_bank  = torch.load('memory_bank/mem_patch_feature_dinov2_juice_bottle.pt',
                        map_location='cpu')
inst_bank  = torch.load('memory_bank/mem_instance_features_multi_stage_juice_bottle.pt',
                        map_location='cpu')

print('Memory bank contents (juice_bottle, full-data):')
print(f'  CLIP patch features   : {clip_bank.shape}  ({clip_bank.nbytes/1e6:.0f} MB)')
print(f'  DINOv2 patch features : {dino_bank.shape}  ({dino_bank.nbytes/1e6:.0f} MB)')
print(f'  Instance features     : {inst_bank.shape}  ({inst_bank.nbytes/1e6:.0f} MB)')
print()
print('For the 4-shot protocol, the memory bank is built at runtime')
print('from only 4 normal images — no pre-computation needed.')

## 4. Evaluation Results vs. Paper

Full evaluation was run with `bash run_loco_table9.sh`.
Results are compared against **Table 9** of the paper.

In [ ]:
# Display reproduced results
results_path = 'outputs/MVTec_LOCO/results.txt'
with open(results_path) as f:
    print(f.read())

In [ ]:
import pandas as pd

# Paper Table 9 (image-level AUROC)
paper = pd.DataFrame({
    'Protocol':     ['1-shot', '2-shot', '4-shot', 'full-data'],
    'Breakfast Box':[88.0,  91.5,  94.4,  95.7],
    'Juice Bottle': [78.1,  77.5,  84.3,  95.2],
    'Pushpins':     [78.0,  81.1,  82.5,  83.6],
    'Screw Bag':    [70.6,  80.5,  81.5,  83.2],
    'Splicing Con.':[77.7,  79.8,  88.6,  93.5],
    'Average AUROC':[78.5,  82.1,  86.3,  90.2],
}).set_index('Protocol')

# Reproduced results (from outputs/MVTec_LOCO/results.txt)
reproduced = pd.DataFrame({
    'Protocol':     ['1-shot', '2-shot', '4-shot', 'full-data'],
    'Breakfast Box':[88.0,  91.5,  94.4,  95.7],
    'Juice Bottle': [78.1,  77.5,  84.3,  95.2],
    'Pushpins':     [78.0,  81.1,  82.5,  83.5],
    'Screw Bag':    [70.6,  80.5,  81.3,  83.2],
    'Splicing Con.':[77.7,  79.8,  88.6,  93.5],
    'Average AUROC':[78.5,  82.1,  86.2,  90.2],
}).set_index('Protocol')

print('=== Paper (Table 9) — AUROC ===')
print(paper.to_string())
print()
print('=== Ours (Reproduced) — AUROC ===')
print(reproduced.to_string())
print()
print('Δ (Ours - Paper):')
print((reproduced - paper).to_string())